# OncoPredict AI: Breast Cancer Prediction Model Training

This interactive notebook guides you through the end-to-end process of training and evaluating our breast tumor classification model.

### Objectives:
1. Load and parse the **Breast Cancer Wisconsin Diagnostic Dataset**.
2. Conduct basic Exploratory Data Analysis (EDA).
3. Preprocess and scale inputs using a standardizer.
4. Train a **Logistic Regression** classifier with hyperparameter tuning.
5. Evaluate clinical performance metrics (Accuracy, F1, Recall, ROC-AUC) and plot heatmaps.
6. Serialize the trained model assets.

In [ ]:
# Cell 1: Environment Setup & Package Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, roc_curve
)

# Set plots backend styles
%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print("All core dependencies loaded successfully!")

## 1. Data Ingestion
We load the Fine Needle Aspirate (FNA) measurements from scikit-learn and format it into a structured DataFrame.

In [ ]:
# Cell 2: Ingest Wisconsin FNA Dataset
raw_data = load_breast_cancer()
df = pd.DataFrame(data=raw_data.data, columns=raw_data.feature_names)

# Sanitize features columns to standard snake_case
df.columns = [col.replace(" ", "_") for col in df.columns]

# Add target diagnostic class (0 = Malignant, 1 = Benign)
df["target"] = raw_data.target

print(f"Ingestion Complete. Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()

## 2. Exploratory Data Analysis (EDA)
Let's look at the distribution of labels and draw a correlation heatmap of core features.

In [ ]:
# Cell 3: Diagnosis Distribution & Correlation
target_counts = df["target"].value_counts()
print(f"Malignant Cases (0): {target_counts[0]} ({target_counts[0]/len(df)*100:.2f}%)")
print(f"Benign Cases (1): {target_counts[1]} ({target_counts[1]/len(df)*100:.2f}%)")

# Plot class imbalance status
plt.figure(figsize=(6, 4))
sns.countplot(x="target", data=df, hue="target", palette=["#e63946", "#2a9d8f"], legend=False)
plt.title("Class Distribution")
plt.xticks(ticks=[0, 1], labels=["Malignant (0)", "Benign (1)"])
plt.show()

# Plot correlation of top 10 mean parameters
mean_cols = [col for col in df.columns if col.startswith("mean_")]
plt.figure(figsize=(10, 8))
sns.heatmap(df[mean_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f", square=True)
plt.title("Core Features Correlation Heatmap")
plt.show()

## 3. Train-Test Split & Data Transformation
We split the data using stratified splits to maintain class distributions, then apply a standard scalar to prevent features with large ranges (e.g. area) from dominating the gradients.

In [ ]:
# Cell 4: Train-Test Split & Fit Scaler
X = df.drop(columns=["target"])
y = df["target"]

# Stratified Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Fit and transform features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training Split features shape: {X_train_scaled.shape}")
print(f"Testing Split features shape: {X_test_scaled.shape}")

## 4. Model Training
We fit a Logistic Regression model with L2 regularization ($C=1.0$) and standard solver parameterizations.

In [ ]:
# Cell 5: Train Logistic Regression Classifier
model = LogisticRegression(C=1.0, penalty="l2", solver="lbfgs", max_iter=1000)
model.fit(X_train_scaled, y_train)

print("Model Training Completed!")
print(f"Intercept: {model.intercept_[0]:.4f}")
print(f"Top 3 positive weights (increase chance of Benign):")
weights = pd.Series(model.coef_[0], index=X.columns).sort_values(ascending=False)
print(weights.head(3))
print("\nTop 3 negative weights (increase chance of Malignant):")
print(weights.tail(3))

## 5. Model Evaluation
Let's compute the accuracy, precision, recall, F1, and AUC scores, and plot the confusion matrix and ROC curves.

In [ ]:
# Cell 6: Metrics and Evaluation
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("=== Logistic Regression Evaluation ===")
print(f"Accuracy:  {accuracy * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall:    {recall * 100:.2f}%")
print(f"F1 Score:  {f1 * 100:.2f}%")
print(f"ROC-AUC:   {auc * 100:.2f}%")

# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Malignant", "Benign"], yticklabels=["Malignant", "Benign"])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# Plot ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC Curve (AUC = {auc:.4f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.legend(loc="lower right")
plt.show()

## 6. Save Artifacts
Finally, we serialize the trained model and standardizer to verify that our serialization handles inference requests correctly.

In [ ]:
# Cell 7: Serialize Objects
os.makedirs("artifacts_experiment", exist_ok=True)

with open("artifacts_experiment/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
with open("artifacts_experiment/model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Experiment assets saved successfully in artifacts_experiment/ directory!")